# MABSA Hotel Bintang 3 — Penggabungan Dataset
## Merge Semua CSV dari Traveloka & Tiket.com

**Notebook ini akan:**
1. Mount Google Drive
2. Scan semua file CSV di folder `Scrap/Traveloka/` dan `Scrap/Tiket/`
3. Gabungkan ke format seragam dengan kolom:
   - `ID_Review` — ID unik per review
   - `Platform` — Traveloka / Tiket
   - `Wilayah` — Nama kota (Bandung, Bogor, dll)
   - `Nama_Hotel` — Nama hotel
   - `Text_Review` — Teks ulasan
   - `Link_Gambar_1` s/d `Link_Gambar_10` — URL gambar
4. Filter hanya review yang punya **teks DAN minimal 1 gambar**
5. Export ke CSV

---

## 1. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Konfigurasi Path

**Sesuaikan path di bawah ini** dengan lokasi folder `Scrap` di Google Drive kamu.

In [3]:
# ============================================================
# KONFIGURASI — SESUAIKAN DI SINI
# ============================================================

# Path ke folder Scrap di Google Drive
# Contoh: jika folder Scrap ada di My Drive/Scrap
BASE_PATH = "/content/drive/MyDrive/Scrap"

# Nama subfolder platform
TRAVELOKA_FOLDER = "Traveloka"
TIKET_FOLDER = "Tiket"

# Nama file output
OUTPUT_FILE = "/content/drive/MyDrive/dataset_mabsa_merged.csv"
OUTPUT_FILE_WITH_IMAGE = "/content/drive/MyDrive/dataset_mabsa_with_image.csv"

# ============================================================

## 3. Scan & Preview Struktur Folder

In [4]:
import os
import pandas as pd
import glob

def scan_folder_structure(base_path):
    """Scan dan tampilkan struktur folder Scrap."""
    print(f"Base path: {base_path}")
    print(f"Folder exists: {os.path.exists(base_path)}\n")

    if not os.path.exists(base_path):
        print("[ERROR] Folder tidak ditemukan!")
        print("Pastikan path BASE_PATH di cell sebelumnya sudah benar.")
        print(f"\nIsi dari /content/drive/MyDrive/:")
        try:
            items = os.listdir("/content/drive/MyDrive/")
            for item in sorted(items)[:20]:
                print(f"  {item}")
        except:
            print("  Tidak bisa membaca folder.")
        return

    total_csv = 0
    for platform in [TRAVELOKA_FOLDER, TIKET_FOLDER]:
        platform_path = os.path.join(base_path, platform)
        if not os.path.exists(platform_path):
            print(f"[WARNING] Folder {platform} tidak ditemukan di {base_path}")
            continue

        print(f"\n📁 {platform}/")
        cities = sorted([d for d in os.listdir(platform_path)
                        if os.path.isdir(os.path.join(platform_path, d))])

        for city in cities:
            city_path = os.path.join(platform_path, city)
            csv_files = [f for f in os.listdir(city_path) if f.endswith('.csv')]
            total_csv += len(csv_files)
            print(f"   📁 {city}/ ({len(csv_files)} hotel)")
            for f in sorted(csv_files):
                size = os.path.getsize(os.path.join(city_path, f)) / 1024
                print(f"      📄 {f} ({size:.0f} KB)")

    print(f"\n{'='*50}")
    print(f"Total file CSV ditemukan: {total_csv}")

scan_folder_structure(BASE_PATH)

Base path: /content/drive/MyDrive/Scrap
Folder exists: True


📁 Traveloka/
   📁 Bandung/ (6 hotel)
      📄 Atlantic City Hotel.csv (210 KB)
      📄 Hay Bandung.csv (291 KB)
      📄 Meize City Center Bandung.csv (172 KB)
      📄 YELLO Hotel Paskal Bandung.csv (498 KB)
      📄 favehotel Premier Cihampelas.csv (289 KB)
      📄 ibis Bandung Trans Studio.csv (324 KB)
   📁 Bekasi/ (4 hotel)
      📄 BATIQA Hotel Jababeka Cikarang.csv (120 KB)
      📄 Hotel Santika Mega City Bekasi.csv (143 KB)
      📄 Yusra Inn Hotel Bekasi.csv (47 KB)
      📄 Zuri Express Lippo Cikarang.csv (50 KB)
   📁 Bogor/ (3 hotel)
      📄 D'Anaya Hotel Bogor.csv (301 KB)
      📄 Hotel Santika Bogor.csv (174 KB)
      📄 Whiz Prime Hotel Pajajaran Bogor.csv (425 KB)
   📁 Cirebon/ (2 hotel)
      📄 Hotel Neo Cirebon by ASTON.csv (176 KB)
      📄 Verse Hotel Cirebon.csv (247 KB)
   📁 Depok/ (3 hotel)
      📄 Hotel Santika Depok.csv (119 KB)
      📄 Savero Hotel Depok.csv (186 KB)
      📄 favehotel Margonda - Depok.csv (89 

## 4. Baca & Gabungkan Semua CSV

In [5]:
import warnings
warnings.filterwarnings('ignore')

# Kolom gambar output (standar 10 kolom)
IMG_COLS_OUTPUT = [f"Link_Gambar_{i}" for i in range(1, 11)]


def read_traveloka_csv(filepath, wilayah, nama_hotel):
    """
    Baca CSV format Traveloka.
    Kolom: teks-ulasan, tanggal, gambar-1-src ... gambar-5-src
    """
    try:
        df = pd.read_csv(filepath, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding='latin-1')
    except Exception as e:
        print(f"  [ERROR] Gagal baca {filepath}: {e}")
        return pd.DataFrame()

    if len(df) == 0:
        return pd.DataFrame()

    # Cari kolom teks (bisa beda format)
    text_col = None
    for candidate in ['teks-ulasan', 'teks_ulasan', 'text', 'review', 'ulasan']:
        if candidate in df.columns:
            text_col = candidate
            break
    # Fallback: cari kolom yang mengandung 'teks' atau 'ulasan'
    if text_col is None:
        for col in df.columns:
            if 'teks' in col.lower() or 'ulasan' in col.lower() or 'review' in col.lower():
                text_col = col
                break

    if text_col is None:
        print(f"  [WARNING] Kolom teks tidak ditemukan di {filepath}")
        print(f"  Kolom yang ada: {list(df.columns)}")
        return pd.DataFrame()

    # Cari kolom gambar Traveloka (gambar-1-src ... gambar-5-src)
    traveloka_img_cols = []
    for i in range(1, 11):
        for pattern in [f'gambar-{i}-src', f'gambar_{i}_src', f'gambar-{i}', f'gambar_{i}']:
            if pattern in df.columns:
                traveloka_img_cols.append(pattern)
                break

    # Bangun dataframe output
    rows = []
    for _, row in df.iterrows():
        text = str(row[text_col]).strip() if pd.notna(row[text_col]) else ""
        if not text or text.lower() == 'nan':
            continue

        img_urls = []
        for col in traveloka_img_cols:
            val = row.get(col, None)
            if pd.notna(val) and str(val).strip() and str(val).strip().lower() != 'nan':
                img_urls.append(str(val).strip())
            else:
                img_urls.append(None)

        # Pad to 10 columns
        while len(img_urls) < 10:
            img_urls.append(None)

        row_data = {
            'Platform': 'Traveloka',
            'Wilayah': wilayah,
            'Nama_Hotel': nama_hotel,
            'Text_Review': text,
        }
        for j, img_col_name in enumerate(IMG_COLS_OUTPUT):
            row_data[img_col_name] = img_urls[j]

        rows.append(row_data)

    return pd.DataFrame(rows)


def read_tiket_csv(filepath, wilayah, nama_hotel):
    """
    Baca CSV format Tiket.com.
    Kolom: url, page, review_index, teks_ulasan, tanggal, gambar-1 ... gambar-10
    """
    try:
        df = pd.read_csv(filepath, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding='latin-1')
    except Exception as e:
        print(f"  [ERROR] Gagal baca {filepath}: {e}")
        return pd.DataFrame()

    if len(df) == 0:
        return pd.DataFrame()

    # Cari kolom teks
    text_col = None
    for candidate in ['teks_ulasan', 'teks-ulasan', 'text', 'review', 'ulasan']:
        if candidate in df.columns:
            text_col = candidate
            break
    if text_col is None:
        for col in df.columns:
            if 'teks' in col.lower() or 'ulasan' in col.lower() or 'review' in col.lower():
                text_col = col
                break

    if text_col is None:
        print(f"  [WARNING] Kolom teks tidak ditemukan di {filepath}")
        print(f"  Kolom yang ada: {list(df.columns)}")
        return pd.DataFrame()

    # Cari kolom gambar Tiket (gambar-1 ... gambar-10)
    tiket_img_cols = []
    for i in range(1, 11):
        for pattern in [f'gambar-{i}', f'gambar_{i}', f'gambar-{i}-src', f'gambar_{i}_src']:
            if pattern in df.columns:
                tiket_img_cols.append(pattern)
                break

    # Bangun dataframe output
    rows = []
    for _, row in df.iterrows():
        text = str(row[text_col]).strip() if pd.notna(row[text_col]) else ""
        if not text or text.lower() == 'nan':
            continue

        img_urls = []
        for col in tiket_img_cols:
            val = row.get(col, None)
            if pd.notna(val) and str(val).strip() and str(val).strip().lower() != 'nan':
                img_urls.append(str(val).strip())
            else:
                img_urls.append(None)

        # Pad to 10 columns
        while len(img_urls) < 10:
            img_urls.append(None)

        row_data = {
            'Platform': 'Tiket',
            'Wilayah': wilayah,
            'Nama_Hotel': nama_hotel,
            'Text_Review': text,
        }
        for j, img_col_name in enumerate(IMG_COLS_OUTPUT):
            row_data[img_col_name] = img_urls[j]

        rows.append(row_data)

    return pd.DataFrame(rows)


print("Fungsi pembaca CSV siap!")

Fungsi pembaca CSV siap!


In [7]:
# ============================================================
# PROSES PENGGABUNGAN
# ============================================================

all_data = []
stats = {'traveloka': 0, 'tiket': 0, 'hotels': 0, 'cities': set(), 'files_read': 0, 'files_error': 0}

for platform_name, folder_name, reader_func in [
    ('Traveloka', TRAVELOKA_FOLDER, read_traveloka_csv),
    ('Tiket', TIKET_FOLDER, read_tiket_csv),
]:
    platform_path = os.path.join(BASE_PATH, folder_name)
    if not os.path.exists(platform_path):
        print(f"[WARNING] Folder {folder_name} tidak ditemukan, skip...")
        continue

    print(f"\n{'='*60}")
    print(f"Memproses: {platform_name}")
    print(f"{'='*60}")

    # Scan kota
    cities = sorted([d for d in os.listdir(platform_path)
                    if os.path.isdir(os.path.join(platform_path, d))])

    for city in cities:
        city_path = os.path.join(platform_path, city)
        csv_files = sorted([f for f in os.listdir(city_path) if f.endswith('.csv')])

        if not csv_files:
            continue

        stats['cities'].add(city)
        print(f"\n  📁 {city} ({len(csv_files)} hotel)")

        for csv_file in csv_files:
            filepath = os.path.join(city_path, csv_file)
            # Nama hotel = nama file tanpa .csv
            nama_hotel = os.path.splitext(csv_file)[0]

            try:
                df_hotel = reader_func(filepath, city, nama_hotel)
                if len(df_hotel) > 0:
                    all_data.append(df_hotel)
                    count = len(df_hotel)
                    if platform_name == 'Traveloka':
                        stats['traveloka'] += count
                    else:
                        stats['tiket'] += count
                    stats['hotels'] += 1
                    stats['files_read'] += 1
                    print(f"     ✓ {nama_hotel}: {count} review")
                else:
                    print(f"     - {nama_hotel}: 0 review (kosong)")
                    stats['files_read'] += 1
            except Exception as e:
                print(f"     ✗ {nama_hotel}: ERROR - {str(e)[:80]}")
                stats['files_error'] += 1

# Gabungkan semua
if all_data:
    df_merged = pd.concat(all_data, ignore_index=True)

    # Tambahkan ID_Review
    df_merged.insert(0, 'ID_Review', range(1, len(df_merged) + 1))

    print(f"\n{'='*60}")
    print(f"HASIL PENGGABUNGAN")
    print(f"{'='*60}")
    print(f"Total review:      {len(df_merged)}")
    print(f"  - Traveloka:     {stats['traveloka']}")
    print(f"  - Tiket.com:     {stats['tiket']}")
    print(f"Total hotel:       {stats['hotels']}")
    print(f"Total kota:        {len(stats['cities'])} ({', '.join(sorted(stats['cities']))})")
    print(f"File dibaca:       {stats['files_read']}")
    print(f"File error:        {stats['files_error']}")
    print(f"\nKolom: {list(df_merged.columns)}")
else:
    print("\n[ERROR] Tidak ada data yang berhasil dibaca!")
    df_merged = pd.DataFrame()


Memproses: Traveloka

  📁 Bandung (6 hotel)
     ✓ Atlantic City Hotel: 244 review
     ✓ Hay Bandung: 283 review
     ✓ Meize City Center Bandung: 186 review
     ✓ YELLO Hotel Paskal Bandung: 455 review
     ✓ favehotel Premier Cihampelas: 320 review
     ✓ ibis Bandung Trans Studio: 382 review

  📁 Bekasi (4 hotel)
     ✓ BATIQA Hotel Jababeka Cikarang: 128 review
     ✓ Hotel Santika Mega City Bekasi: 149 review
     ✓ Yusra Inn Hotel Bekasi: 62 review
     ✓ Zuri Express Lippo Cikarang: 56 review

  📁 Bogor (3 hotel)
     ✓ D'Anaya Hotel Bogor: 322 review
     ✓ Hotel Santika Bogor: 180 review
     ✓ Whiz Prime Hotel Pajajaran Bogor: 466 review

  📁 Cirebon (2 hotel)
     ✓ Hotel Neo Cirebon by ASTON: 199 review
     ✓ Verse Hotel Cirebon: 260 review

  📁 Depok (3 hotel)
     ✓ Hotel Santika Depok: 114 review
     ✓ Savero Hotel Depok: 195 review
     ✓ favehotel Margonda - Depok: 93 review

  📁 Garut (2 hotel)
     ✓ Hotel Tirta Kencana Cipanas Garut: 627 review
     ✓ favehotel

## 5. Preview Data Gabungan

In [8]:
if len(df_merged) > 0:
    # Preview 10 baris pertama
    print("Preview 10 baris pertama:")
    display(df_merged.head(10))

    # Statistik per platform
    print(f"\n{'='*50}")
    print("Distribusi per Platform:")
    print(df_merged['Platform'].value_counts())

    # Statistik per wilayah
    print(f"\nDistribusi per Wilayah:")
    print(df_merged['Wilayah'].value_counts())

    # Statistik per hotel (top 20)
    print(f"\nTop 20 Hotel (jumlah review):")
    print(df_merged['Nama_Hotel'].value_counts().head(20))

    # Hitung jumlah gambar per review
    img_cols = [f'Link_Gambar_{i}' for i in range(1, 11)]
    df_merged['_jumlah_gambar'] = df_merged[img_cols].notna().sum(axis=1)

    print(f"\nDistribusi jumlah gambar per review:")
    print(df_merged['_jumlah_gambar'].value_counts().sort_index())

    has_image = (df_merged['_jumlah_gambar'] > 0).sum()
    no_image = (df_merged['_jumlah_gambar'] == 0).sum()
    print(f"\nReview DENGAN gambar: {has_image} ({has_image/len(df_merged)*100:.1f}%)")
    print(f"Review TANPA gambar:  {no_image} ({no_image/len(df_merged)*100:.1f}%)")

Preview 10 baris pertama:


,ID_Review,Platform,Wilayah,Nama_Hotel,Text_Review,Link_Gambar_1,Link_Gambar_2,Link_Gambar_3,Link_Gambar_4,Link_Gambar_5,Link_Gambar_6,Link_Gambar_7,Link_Gambar_8,Link_Gambar_9,Link_Gambar_10
0,1,Traveloka,Bandung,Atlantic City Hotel,Pesan mendadak hotel di daerah Kota Bandung sa...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,None,None,None,None,None,None,None,None,None
1,2,Traveloka,Bandung,Atlantic City Hotel,Menyenangkan menginap di Hotel Atlantic. Hotel...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,None,None,None,None,None
2,3,Traveloka,Bandung,Atlantic City Hotel,"Bersih, rapi, cukup strategis dan sudah profes...",https://d24fi85k5ai89e.cloudfront.net/REVIEW_V...,None,None,None,None,None,None,None,None,None
3,4,Traveloka,Bandung,Atlantic City Hotel,"Mantap, oke, dekat dengan semuanya, di lantai ...",https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,None,None,None,None,None,None,None,None,None
4,5,Traveloka,Bandung,Atlantic City Hotel,Staff sangat ramah dan sangat membantu. Walaup...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,None,None,None,None,None,None
5,6,Traveloka,Bandung,Atlantic City Hotel,"Staff hotel ramah-ramah semua, proses cek in d...",https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,None,None,None,None,None,None,None,None,None
6,7,Traveloka,Bandung,Atlantic City Hotel,"Staf hotel ramah. Lokasi sebelum KFC, menyeber...",https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://d24fi85k5ai89e.cloudfront.net/REVIEW_V...,None,None,None,None,None,None,None,None
7,8,Traveloka,Bandung,Atlantic City Hotel,"Staff ramah, minta hairdryer langsung dikasih....",https://d24fi85k5ai89e.cloudfront.net/REVIEW_V...,None,None,None,None,None,None,None,None,None
8,9,Traveloka,Bandung,Atlantic City Hotel,"staff ramah, harga murah tapi ga murahan 👍",https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,None,None,None,None,None,None,None,None,None
9,10,Traveloka,Bandung,Atlantic City Hotel,Pengalaman liburan yang menyenangkan.... semua...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,None,None,None,None,None,None,None



Distribusi per Platform:
Platform
Traveloka    5952
Tiket        2146
Name: count, dtype: int64

Distribusi per Wilayah:
Wilayah
Bandung        2425
Bogor          1455
Pangandaran    1106
Garut          1015
Depok           614
Cirebon         575
Bekasi          568
Sukabumi        252
Subang           88
Name: count, dtype: int64

Top 20 Hotel (jumlah review):
Nama_Hotel
Hotel Tirta Kencana Cipanas Garut    836
Whiz Prime Hotel Pajajaran Bogor     720
Laut Biru Resort Hotel               709
YELLO Hotel Paskal Bandung           636
ibis Bandung Trans Studio            534
favehotel Premier Cihampelas         401
D'Anaya Hotel Bogor                  322
Hay Bandung                          319
Verse Hotel Cirebon                  314
Savero Hotel Depok                   295
Atlantic City Hotel                  288
Hotel Neo Cirebon by ASTON           261
Hotel Santika Bogor                  260
Meize City Center Bandung            247
Hotel Santika Mega City Bekasi       204
Sun In 

## 6. Filter: Hanya Review dengan Teks + Gambar

Karena ini penelitian **MABSA** (multimodal), hanya simpan review yang punya **teks DAN minimal 1 gambar**.

In [9]:
if len(df_merged) > 0:
    img_cols = [f'Link_Gambar_{i}' for i in range(1, 11)]

    # Filter: harus punya teks yang valid
    has_text = df_merged['Text_Review'].notna() & (df_merged['Text_Review'].str.strip() != '')

    # Filter: harus punya minimal 1 gambar
    has_img = df_merged[img_cols].notna().any(axis=1)

    # Gabungkan filter
    df_filtered = df_merged[has_text & has_img].copy()

    # Reset ID_Review setelah filter
    df_filtered['ID_Review'] = range(1, len(df_filtered) + 1)

    # Hapus kolom helper
    if '_jumlah_gambar' in df_filtered.columns:
        df_filtered = df_filtered.drop(columns=['_jumlah_gambar'])

    print(f"{'='*50}")
    print(f"HASIL FILTER (Teks + Gambar)")
    print(f"{'='*50}")
    print(f"Sebelum filter: {len(df_merged)} review")
    print(f"Setelah filter: {len(df_filtered)} review")
    print(f"Dihapus:        {len(df_merged) - len(df_filtered)} review (tanpa gambar)")

    print(f"\nDistribusi per Platform (setelah filter):")
    print(df_filtered['Platform'].value_counts())

    print(f"\nDistribusi per Wilayah (setelah filter):")
    print(df_filtered['Wilayah'].value_counts())

    # Hitung ulang jumlah gambar
    jumlah_gambar = df_filtered[img_cols].notna().sum(axis=1)
    total_gambar = int(jumlah_gambar.sum())
    print(f"\nTotal gambar: {total_gambar}")
    print(f"Rata-rata gambar per review: {jumlah_gambar.mean():.1f}")

    print(f"\nPreview 5 baris pertama:")
    display(df_filtered.head())

HASIL FILTER (Teks + Gambar)
Sebelum filter: 8098 review
Setelah filter: 8081 review
Dihapus:        17 review (tanpa gambar)

Distribusi per Platform (setelah filter):
Platform
Traveloka    5949
Tiket        2132
Name: count, dtype: int64

Distribusi per Wilayah (setelah filter):
Wilayah
Bandung        2418
Bogor          1451
Pangandaran    1104
Garut          1014
Depok           613
Cirebon         574
Bekasi          568
Sukabumi        251
Subang           88
Name: count, dtype: int64

Total gambar: 17483
Rata-rata gambar per review: 2.2

Preview 5 baris pertama:


,ID_Review,Platform,Wilayah,Nama_Hotel,Text_Review,Link_Gambar_1,Link_Gambar_2,Link_Gambar_3,Link_Gambar_4,Link_Gambar_5,Link_Gambar_6,Link_Gambar_7,Link_Gambar_8,Link_Gambar_9,Link_Gambar_10
0,1,Traveloka,Bandung,Atlantic City Hotel,Pesan mendadak hotel di daerah Kota Bandung sa...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,None,None,None,None,None,None,None,None,None
1,2,Traveloka,Bandung,Atlantic City Hotel,Menyenangkan menginap di Hotel Atlantic. Hotel...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,None,None,None,None,None
2,3,Traveloka,Bandung,Atlantic City Hotel,"Bersih, rapi, cukup strategis dan sudah profes...",https://d24fi85k5ai89e.cloudfront.net/REVIEW_V...,None,None,None,None,None,None,None,None,None
3,4,Traveloka,Bandung,Atlantic City Hotel,"Mantap, oke, dekat dengan semuanya, di lantai ...",https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,None,None,None,None,None,None,None,None,None
4,5,Traveloka,Bandung,Atlantic City Hotel,Staff sangat ramah dan sangat membantu. Walaup...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,https://ik.imagekit.io/tvlk/ugc-review/guys1L+...,None,None,None,None,None,None


## 7. Cek Duplikat

In [10]:
if len(df_filtered) > 0:
    # Cek duplikat berdasarkan teks review
    duplicates = df_filtered.duplicated(subset=['Text_Review'], keep=False)
    n_dup = duplicates.sum()

    print(f"Review dengan teks duplikat: {n_dup}")

    if n_dup > 0:
        # Tampilkan contoh duplikat
        dup_df = df_filtered[duplicates].sort_values('Text_Review')
        print(f"\nContoh duplikat (5 pertama):")
        for i, (_, row) in enumerate(dup_df.head(10).iterrows()):
            print(f"  [{row['Platform']}] {row['Nama_Hotel']}: {row['Text_Review'][:80]}...")
            if (i + 1) % 2 == 0:
                print()

        # Hapus duplikat (simpan yang pertama)
        df_before_dedup = len(df_filtered)
        df_filtered = df_filtered.drop_duplicates(subset=['Text_Review'], keep='first').copy()
        df_filtered['ID_Review'] = range(1, len(df_filtered) + 1)
        print(f"\nSetelah hapus duplikat: {len(df_filtered)} review (dihapus {df_before_dedup - len(df_filtered)})")
    else:
        print("Tidak ada duplikat.")

Review dengan teks duplikat: 32

Contoh duplikat (5 pertama):
  [Tiket] Surya Kencana Seaside Hotel: Bagus...
  [Tiket] Hotel Tirta Kencana Cipanas Garut: Bagus...

  [Tiket] Hotel Tirta Kencana Cipanas Garut: Bagus...
  [Tiket] ibis Bandung Trans Studio: Bagus...

  [Tiket] Hay Bandung: Bukan saya yang menginap tapi tidak ada komplain...
  [Tiket] Meize City Center Bandung: Bukan saya yang menginap tapi tidak ada komplain...

  [Tiket] YELLO Hotel Paskal Bandung: Bukan saya yang menginap tapi tidak ada komplain...
  [Tiket] YELLO Hotel Paskal Bandung: Bukan saya yang menginap tapi tidak ada komplain...

  [Tiket] Hotel Tirta Kencana Cipanas Garut: Hotel yang cocok untuk keluarga, ada air hangatnya, disarankan renang pagi2. Ada...
  [Tiket] Hotel Tirta Kencana Cipanas Garut: Hotel yang cocok untuk keluarga, ada air hangatnya, disarankan renang pagi2. Ada...


Setelah hapus duplikat: 8061 review (dihapus 20)


## 8. Export ke CSV

In [11]:
if len(df_filtered) > 0:
    # Pastikan kolom helper tidak ikut
    if '_jumlah_gambar' in df_merged.columns:
        df_all = df_merged.drop(columns=['_jumlah_gambar'])
    else:
        df_all = df_merged.copy()

    # Simpan SEMUA review (termasuk yang tanpa gambar)
    df_all.to_csv(OUTPUT_FILE, index=False)
    print(f"File semua review disimpan: {OUTPUT_FILE}")
    print(f"  Jumlah: {len(df_all)} review")

    # Simpan HANYA yang punya gambar (untuk MABSA)
    df_filtered.to_csv(OUTPUT_FILE_WITH_IMAGE, index=False)
    print(f"\nFile review + gambar disimpan: {OUTPUT_FILE_WITH_IMAGE}")
    print(f"  Jumlah: {len(df_filtered)} review")

    # Download ke komputer lokal
    from google.colab import files
    print(f"\nMengunduh file ke komputer lokal...")
    files.download(OUTPUT_FILE_WITH_IMAGE)

    print(f"\n{'='*50}")
    print(f"SELESAI!")
    print(f"{'='*50}")
    print(f"File utama untuk MABSA: {OUTPUT_FILE_WITH_IMAGE}")
    print(f"Total: {len(df_filtered)} pasangan teks+gambar")
    print(f"\nFile ini siap untuk:")
    print(f"  1. Auto-labeling teks  (mabsa_auto_labeling_ollama.py)")
    print(f"  2. Auto-labeling gambar (mabsa_auto_labeling_image_ollama.py)")

File semua review disimpan: /content/drive/MyDrive/dataset_mabsa_merged.csv
  Jumlah: 8098 review

File review + gambar disimpan: /content/drive/MyDrive/dataset_mabsa_with_image.csv
  Jumlah: 8061 review

Mengunduh file ke komputer lokal...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


SELESAI!
File utama untuk MABSA: /content/drive/MyDrive/dataset_mabsa_with_image.csv
Total: 8061 pasangan teks+gambar

File ini siap untuk:
  1. Auto-labeling teks  (mabsa_auto_labeling_ollama.py)
  2. Auto-labeling gambar (mabsa_auto_labeling_image_ollama.py)


## 9. Ringkasan Final

In [12]:
if len(df_filtered) > 0:
    print(f"{'='*60}")
    print(f"RINGKASAN DATASET MABSA")
    print(f"{'='*60}")
    print(f"")
    print(f"Total review (teks + gambar): {len(df_filtered)}")
    print(f"Target: 7.000 pasangan")
    print(f"Progress: {len(df_filtered)/7000*100:.1f}%")
    print(f"")
    print(f"Platform:")
    for platform, count in df_filtered['Platform'].value_counts().items():
        print(f"  {platform}: {count} review ({count/len(df_filtered)*100:.1f}%)")
    print(f"")
    print(f"Wilayah:")
    for wilayah, count in df_filtered['Wilayah'].value_counts().items():
        print(f"  {wilayah}: {count} review ({count/len(df_filtered)*100:.1f}%)")
    print(f"")
    print(f"Hotel unik: {df_filtered['Nama_Hotel'].nunique()}")
    print(f"")

    img_cols = [f'Link_Gambar_{i}' for i in range(1, 11)]
    total_imgs = int(df_filtered[img_cols].notna().sum().sum())
    print(f"Total gambar: {total_imgs}")
    print(f"Rata-rata gambar/review: {total_imgs/len(df_filtered):.1f}")

RINGKASAN DATASET MABSA

Total review (teks + gambar): 8061
Target: 7.000 pasangan
Progress: 115.2%

Platform:
  Traveloka: 5948 review (73.8%)
  Tiket: 2113 review (26.2%)

Wilayah:
  Bandung: 2410 review (29.9%)
  Bogor: 1444 review (17.9%)
  Pangandaran: 1102 review (13.7%)
  Garut: 1011 review (12.5%)
  Depok: 613 review (7.6%)
  Cirebon: 574 review (7.1%)
  Bekasi: 568 review (7.0%)
  Sukabumi: 251 review (3.1%)
  Subang: 88 review (1.1%)

Hotel unik: 29

Total gambar: 17453
Rata-rata gambar/review: 2.2
